# Telco Next Best Offer — Multiclass Classification

**Mục tiêu**: Dự đoán cột `Offers` (5 nhãn) dựa trên đặc điểm và hành vi sử dụng của khách hàng viễn thông, phục vụ cross-selling.

**Hướng dẫn sử dụng**:
1. Chạy cell đầu tiên để upload file CSV `DR_Demo_Telco_Next_Best_Offer_Multiclass.csv`
2. Chạy tuần tự từng cell từ trên xuống dưới
3. Xem kết quả đánh giá ở cuối notebook

## Bước 0: Cài đặt thư viện & Upload dữ liệu

In [ ]:
!pip install -q xgboost scikit-learn pandas matplotlib seaborn

from google.colab import files
uploaded = files.upload()  # Chọn file DR_Demo_Telco_Next_Best_Offer_Multiclass.csv
filename = list(uploaded.keys())[0]
print(f"Đã upload: {filename}")

## Bước 1: Đọc dữ liệu & Khám phá sơ bộ (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(filename)

# Đổi tên cột từ dấu chấm (.) sang gạch dưới (_) để đồng nhất với chuẩn JSON/API
# (JSON/Pydantic không cho phép dấu chấm trong tên trường, nên đổi ngay từ đầu
# giúp tránh phải chuyển đổi qua lại phức tạp lúc deploy về sau)
df.columns = df.columns.str.replace('.', '_', regex=False)

print("Kích thước dữ liệu:", df.shape)
df.head()

In [ ]:
# Kiểm tra missing values
print("Missing values:\n", df.isna().sum())
print("\nPhân bố nhãn (Offers):")
print(df['Offers'].value_counts())

plt.figure(figsize=(8,4))
sns.countplot(data=df, x='Offers', order=df['Offers'].value_counts().index)
plt.xticks(rotation=30, ha='right')
plt.title('Phân bố các nhãn Offers')
plt.tight_layout()
plt.show()

## Bước 2: Tiền xử lý dữ liệu (phần cấu trúc)

- Loại bỏ `Cust_ID` (chỉ là định danh)
- Tách riêng `tariff_plan_conds` để xử lý NLP ở bước sau
- Mã hóa các biến phân loại (`State`, `Area_code`, `International_plan`, `Voice_mail_plan`)

In [ ]:
# Tách cột text ra riêng để xử lý NLP ở Bước 2.5
text_data = df['tariff_plan_conds'].fillna('')

df_model = df.drop(columns=['Cust_ID', 'tariff_plan_conds'])

# Binary encoding
df_model['International_plan'] = df_model['International_plan'].map({'Yes': 1, 'No': 0})
df_model['Voice_mail_plan'] = df_model['Voice_mail_plan'].map({'Yes': 1, 'No': 0})

# One-hot encoding cho State và Area_code
df_model = pd.get_dummies(df_model, columns=['State', 'Area_code'], drop_first=True)

X_structured = df_model.drop(columns=['Offers'])
y = df_model['Offers']
structured_cols = list(X_structured.columns)

print("Số lượng features cấu trúc sau encoding:", X_structured.shape[1])
X_structured.head()

## Bước 2.5: Xử lý văn bản (NLP) cho cột `tariff_plan_conds`

Cột `tariff_plan_conds` là văn bản tự do mô tả điều khoản/quy tắc tính cước (376 nội dung khác nhau). Thay vì bỏ qua, ta trích xuất đặc trưng bằng **TF-IDF** (Term Frequency – Inverse Document Frequency):

- Loại bỏ stopwords tiếng Anh
- Dùng unigram + bigram (`ngram_range=(1,2)`) để bắt được cụm từ như "Suspension Charges", "Usage Charges"
- Giới hạn `max_features=100` để tránh bùng nổ chiều dữ liệu
- Kết quả là ma trận thưa (sparse matrix) — mỗi cột là trọng số TF-IDF của một từ/cụm từ

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=100,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(text_data)
tfidf_feature_names = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]

print("Số chiều TF-IDF:", tfidf_matrix.shape[1])
print("Ví dụ một số từ/cụm từ quan trọng nhất:", tfidf_feature_names[:15])

In [ ]:
# Kết hợp features cấu trúc (dense) + features TF-IDF (sparse) thành một ma trận thưa
from scipy.sparse import hstack, csr_matrix

# Ép kiểu float vì DataFrame chứa cả int/bool (từ get_dummies) -> .values có thể ra dtype object
X_combined = hstack([csr_matrix(X_structured.values.astype(float)), tfidf_matrix]).tocsr()
combined_feature_names = list(X_structured.columns) + tfidf_feature_names

print("Tổng số features sau khi kết hợp (cấu trúc + TF-IDF):", X_combined.shape[1])

## Bước 3: Chia tập Train/Test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Ánh xạ nhãn:", dict(zip(le.classes_, range(len(le.classes_)))))

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# with_mean=False vì dữ liệu là sparse matrix (bắt buộc cho Logistic Regression)
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

## Bước 4: Huấn luyện & So sánh 3 mô hình

- Logistic Regression (baseline đơn giản)
- Random Forest
- XGBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=300, random_state=42, eval_metric='mlogloss')
}

results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'accuracy': acc}
    print(f"{name}: Accuracy = {acc:.4f}")

## Bước 5: Đánh giá chi tiết mô hình tốt nhất

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_preds = results[best_name]['preds']
print(f"=== Mô hình tốt nhất: {best_name} ===\n")
print(classification_report(y_test, best_preds, target_names=le.classes_))

cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title(f'Confusion Matrix - {best_name}')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Bước 6: Feature Importance (mức độ ảnh hưởng của từng biến, bao gồm cả đặc trưng TF-IDF)

In [ ]:
if best_name in ['Random Forest', 'XGBoost']:
    importances = results[best_name]['model'].feature_importances_
    feat_imp = pd.Series(importances, index=combined_feature_names).sort_values(ascending=False).head(15)

    plt.figure(figsize=(8,6))
    feat_imp.plot(kind='barh')
    plt.gca().invert_yaxis()
    plt.title(f'Top 15 Feature Importance - {best_name}')
    plt.xlabel('Mức độ ảnh hưởng')
    plt.tight_layout()
    plt.show()

    # Xem riêng các đặc trưng TF-IDF (từ cột tariff_plan_conds) có đóng góp ra sao
    tfidf_imp = feat_imp[feat_imp.index.str.startswith('tfidf_')]
    print("\nSố đặc trưng TF-IDF nằm trong Top 15:", len(tfidf_imp))
    if len(tfidf_imp) > 0:
        print(tfidf_imp)
else:
    print("Feature importance chỉ áp dụng cho Random Forest / XGBoost.")

## Bước 7 (tùy chọn): Lưu mô hình để sử dụng sau

In [ ]:
import joblib, shutil

final_model = results[best_name]['model']

joblib.dump(final_model, 'best_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(combined_feature_names, 'feature_columns.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(structured_cols, 'structured_columns.pkl')

# Gom tất cả vào 1 file zip duy nhất cho dễ lưu trữ / tải lên lại sau này
import os
os.makedirs('model_artifacts', exist_ok=True)
for fname in ['best_model.pkl', 'label_encoder.pkl', 'feature_columns.pkl',
              'tfidf_vectorizer.pkl', 'structured_columns.pkl']:
    shutil.copy(fname, f'model_artifacts/{fname}')

shutil.make_archive('model_artifacts', 'zip', 'model_artifacts')

from google.colab import files
files.download('model_artifacts.zip')
print("Đã lưu mô hình vào 'model_artifacts.zip'. Giữ file này lại — dùng ở Bước 7.5 để khôi phục mà không cần train lại.")


## Bước 7.5: Khôi phục mô hình đã lưu (bỏ qua nếu vừa chạy xong Bước 0–7 trong phiên này)

**Dùng khi nào**: Runtime Colab bị ngắt/reset, hoặc bạn mở lại notebook ở phiên mới và **không muốn chạy lại từ Bước 0 đến Bước 7** (tốn thời gian train lại mô hình).

**Cách dùng**: Chạy cell bên dưới, upload file `model_artifacts.zip` đã tải về ở Bước 7 lần chạy trước. Sau khi chạy xong cell này, bạn có thể nhảy thẳng tới **Bước 8** hoặc **Bước 9** mà không cần chạy Bước 0–7.

In [ ]:
import joblib, zipfile, os, shutil
from google.colab import files

print("Upload file 'model_artifacts.zip' (nếu có), HOẶC chọn nhiều file .pkl cùng lúc")
print("(giữ Ctrl/Cmd khi chọn để chọn nhiều file): best_model.pkl, label_encoder.pkl, tfidf_vectorizer.pkl, structured_columns.pkl\n")
restore_uploaded = files.upload()

restore_dir = 'model_artifacts_restored'
os.makedirs(restore_dir, exist_ok=True)

for fname in restore_uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(restore_dir)
    else:
        shutil.copy(fname, f'{restore_dir}/{fname}')

# Kiểm tra đủ file bắt buộc trước khi load, báo rõ file nào còn thiếu
required_files = ['best_model.pkl', 'label_encoder.pkl', 'tfidf_vectorizer.pkl', 'structured_columns.pkl']

available = os.listdir(restore_dir)
missing = [f for f in required_files if f not in available]

if missing:
    print(f"THIẾU FILE: {missing}")
    print(f"Các file đã có trong '{restore_dir}': {available}")
    print("\n=> Vui lòng quay lại Bước 7, chạy lại cell để tải đúng 1 file 'model_artifacts.zip' đầy đủ,")
    print("   HOẶC upload thêm các file .pkl còn thiếu ở trên rồi chạy lại cell này.")
else:
    final_model = joblib.load(f'{restore_dir}/best_model.pkl')
    le = joblib.load(f'{restore_dir}/label_encoder.pkl')
    tfidf = joblib.load(f'{restore_dir}/tfidf_vectorizer.pkl')
    structured_cols = joblib.load(f'{restore_dir}/structured_columns.pkl')

    print("Đã khôi phục mô hình thành công. Có thể dùng ngay Bước 8 hoặc Bước 9.")
    print("Các nhãn Offer:", list(le.classes_))


## Bước 8: Áp dụng mô hình cho khách hàng mới (Inference)

Có 2 cách dự đoán:
- **8.1**: Nhập tay thông tin **một khách hàng** để dự đoán nhanh
- **8.2**: **Upload file CSV** chứa nhiều khách hàng (chưa có cột `Offers`) để dự đoán hàng loạt

Cả 2 đều dùng `final_model`, `le`, `tfidf`, `structured_cols` — các biến này có sẵn sau khi chạy xong Bước 7, **hoặc** sau khi khôi phục ở Bước 7.5 (không cần chạy lại từ đầu).

### 8.1. Dự đoán 1 khách hàng (nhập tay)

In [ ]:
# ===== NHẬP THÔNG TIN KHÁCH HÀNG MẪU TẠI ĐÂY =====
sample_customer = {
    'State': 'NJ',
    'Account_length': 100,
    'Area_code': 415,
    'International_plan': 'No',
    'Voice_mail_plan': 'Yes',
    'Number_vmail_messages': 20,
    'Total_day_minutes': 200.0,
    'Total_day_calls': 100,
    'Total_day_charge': 34.0,
    'Total_eve_minutes': 180.0,
    'Total_eve_calls': 95,
    'Total_eve_charge': 15.3,
    'Total_night_minutes': 210.0,
    'Total_night_calls': 100,
    'Total_night_charge': 9.5,
    'Total_intl_minutes': 10.0,
    'Total_intl_calls': 4,
    'Total_intl_charge': 2.7,
    'Customer_service_calls': 1,
    'tariff_plan_conds': 'Product Usage Charges: charge based on usage of the service.'
}
# ==================================================

def predict_offer(customer_dict, model, tfidf_vectorizer, structured_cols, label_encoder):
    """Dự đoán Offer cho một khách hàng mới (dict thông tin thô, chưa encoding)."""
    text = customer_dict.get('tariff_plan_conds', '')
    raw = {k: v for k, v in customer_dict.items() if k != 'tariff_plan_conds'}

    row = pd.DataFrame([raw])
    row['International_plan'] = row['International_plan'].map({'Yes': 1, 'No': 0})
    row['Voice_mail_plan'] = row['Voice_mail_plan'].map({'Yes': 1, 'No': 0})
    row = pd.get_dummies(row, columns=['State', 'Area_code'])
    row = row.reindex(columns=structured_cols, fill_value=0)

    text_vec = tfidf_vectorizer.transform([text])

    from scipy.sparse import hstack, csr_matrix
    combined = hstack([csr_matrix(row.values.astype(float)), text_vec]).tocsr()

    pred_idx = model.predict(combined)[0]
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(combined)[0]
        proba_df = pd.Series(proba, index=label_encoder.classes_).sort_values(ascending=False)
        return pred_label, proba_df
    return pred_label, None

predicted_offer, proba = predict_offer(sample_customer, final_model, tfidf, structured_cols, le)

print("=== Kết quả dự đoán ===")
print(f"Offer được đề xuất: {predicted_offer}\n")
if proba is not None:
    print("Xác suất chi tiết theo từng nhãn:")
    print(proba)


### 8.2. Dự đoán hàng loạt bằng cách upload file CSV

File CSV cần có **đầy đủ các cột đầu vào giống dữ liệu gốc** (State, Account_length, International_plan, Voice_mail_plan, Total_day_minutes, ..., tariff_plan_conds) — chỉ khác là **không có cột `Offers`** (vì đây là cột cần dự đoán).

In [ ]:
from google.colab import files

print("Vui lòng upload file CSV khách hàng mới (chưa có cột Offers):")
new_uploaded = files.upload()
new_filename = list(new_uploaded.keys())[0]

new_df = pd.read_csv(new_filename)
print("Kích thước file mới:", new_df.shape)
new_df.head()


In [ ]:
# Xử lý dữ liệu mới đúng pipeline như lúc train, rồi dự đoán bằng final_model
new_df.columns = new_df.columns.str.replace('.', '_', regex=False)
new_text = new_df['tariff_plan_conds'].fillna('') if 'tariff_plan_conds' in new_df.columns else pd.Series([''] * len(new_df))

drop_cols = [c for c in ['Cust_ID', 'tariff_plan_conds', 'Offers'] if c in new_df.columns]
new_structured = new_df.drop(columns=drop_cols)

new_structured['International_plan'] = new_structured['International_plan'].map({'Yes': 1, 'No': 0})
new_structured['Voice_mail_plan'] = new_structured['Voice_mail_plan'].map({'Yes': 1, 'No': 0})
new_structured = pd.get_dummies(new_structured, columns=['State', 'Area_code'])

# Căn chỉnh đúng cột như lúc train (cột thiếu -> 0, cột thừa -> bỏ)
new_structured = new_structured.reindex(columns=structured_cols, fill_value=0)

new_text_vec = tfidf.transform(new_text)

from scipy.sparse import hstack, csr_matrix
new_combined = hstack([csr_matrix(new_structured.values.astype(float)), new_text_vec]).tocsr()

new_pred_idx = final_model.predict(new_combined)
new_pred_labels = le.inverse_transform(new_pred_idx)

result_df = new_df.copy()
result_df['Offer_du_doan'] = new_pred_labels

if hasattr(final_model, 'predict_proba'):
    new_proba = final_model.predict_proba(new_combined)
    for i, c in enumerate(le.classes_):
        result_df[f"XacSuat_{c}"] = new_proba[:, i]

result_df.to_csv('ket_qua_du_doan_moi.csv', index=False, encoding='utf-8-sig')
print(result_df.head(10))

files.download('ket_qua_du_doan_moi.csv')
print("\nĐã tải file 'ket_qua_du_doan_moi.csv' về máy.")


## Bước 9: Xuất kết quả dự đoán trên tập test ra CSV

So sánh Offer thực tế vs Offer dự đoán trên tập test, kèm xác suất từng nhãn — dùng để kiểm tra chất lượng mô hình.

**Lưu ý**: Bước này cần `X_test`, `y_test` — chỉ chạy được nếu bạn vừa train xong ở phiên hiện tại (Bước 0–4), **không dùng được nếu chỉ khôi phục mô hình ở Bước 7.5** (vì khi đó không có sẵn tập test).

In [ ]:
test_pred_idx = final_model.predict(X_test)
test_pred_labels = le.inverse_transform(test_pred_idx)
test_actual_labels = le.inverse_transform(y_test)

output_df = pd.DataFrame({
    'Offer_thuc_te': test_actual_labels,
    'Offer_du_doan': test_pred_labels,
    'Du_doan_dung': test_actual_labels == test_pred_labels
})

if hasattr(final_model, 'predict_proba'):
    proba_matrix = final_model.predict_proba(X_test)
    proba_cols = pd.DataFrame(proba_matrix, columns=[f"XacSuat_{c}" for c in le.classes_])
    output_df = pd.concat([output_df.reset_index(drop=True), proba_cols], axis=1)

output_df.to_csv('ket_qua_du_doan_test.csv', index=False, encoding='utf-8-sig')
print(f"Độ chính xác trên tập test: {output_df['Du_doan_dung'].mean():.2%}")
print(output_df.head(10))

from google.colab import files
files.download('ket_qua_du_doan_test.csv')
print("\nĐã tải file 'ket_qua_du_doan_test.csv' về máy.")


## Bước 10: Đóng gói mô hình để triển khai (Deploy lên Render)

Bước này gom toàn bộ file cần thiết (mô hình, encoder, vectorizer, danh sách cột) + một **API dự đoán bằng FastAPI** (`app.py`) + `requirements.txt` + `Dockerfile` + `README.md` vào một thư mục, nén thành file `.zip` duy nhất để tải về.

**Sau khi tải về**: giải nén → push lên GitHub repo → tạo Web Service mới trên Render, trỏ tới repo. Render tự động nhận diện `Dockerfile` và build/chạy — không cần khai báo Build Command/Start Command thủ công.

In [ ]:
import os, json, joblib, shutil

deploy_dir = 'telco_nbo_deploy'
os.makedirs(deploy_dir, exist_ok=True)

# Dùng final_model / structured_cols đã có sẵn (từ Bước 7 hoặc từ Bước 7.5 nếu khôi phục)
joblib.dump(final_model, f'{deploy_dir}/model.pkl')
joblib.dump(le, f'{deploy_dir}/label_encoder.pkl')
joblib.dump(tfidf, f'{deploy_dir}/tfidf_vectorizer.pkl')
joblib.dump(structured_cols, f'{deploy_dir}/structured_columns.pkl')

with open(f'{deploy_dir}/model_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'labels': list(le.classes_)
    }, f, ensure_ascii=False, indent=2)

print(f"Đã lưu artifacts vào thư mục '{deploy_dir}/'")

In [ ]:
# Sinh file app.py - API FastAPI để deploy (Render)
app_code = '''
from fastapi import FastAPI, UploadFile, File, Query
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import joblib
import pandas as pd
import io
from scipy.sparse import hstack, csr_matrix

app = FastAPI(title="Telco Next Best Offer API")

model = joblib.load("model.pkl")
label_encoder = joblib.load("label_encoder.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")
structured_cols = joblib.load("structured_columns.pkl")


class Customer(BaseModel):
    State: str
    Account_length: int
    Area_code: int
    International_plan: str
    Voice_mail_plan: str
    Number_vmail_messages: int
    Total_day_minutes: float
    Total_day_calls: int
    Total_day_charge: float
    Total_eve_minutes: float
    Total_eve_calls: int
    Total_eve_charge: float
    Total_night_minutes: float
    Total_night_calls: int
    Total_night_charge: float
    Total_intl_minutes: float
    Total_intl_calls: int
    Total_intl_charge: float
    Customer_service_calls: int
    tariff_plan_conds: str = ""


def preprocess_and_predict(df: pd.DataFrame):
    """Xu ly 1 hoac nhieu dong khach hang (DataFrame) va tra ve du doan."""
    df = df.copy()
    df.columns = df.columns.str.replace(".", "_", regex=False)

    text = df["tariff_plan_conds"].fillna("") if "tariff_plan_conds" in df.columns else pd.Series([""] * len(df))

    drop_cols = [c for c in ["Cust_ID", "tariff_plan_conds", "Offers"] if c in df.columns]
    structured = df.drop(columns=drop_cols)

    structured["International_plan"] = structured["International_plan"].map({"Yes": 1, "No": 0})
    structured["Voice_mail_plan"] = structured["Voice_mail_plan"].map({"Yes": 1, "No": 0})
    structured = pd.get_dummies(structured, columns=["State", "Area_code"])
    structured = structured.reindex(columns=structured_cols, fill_value=0)

    text_vec = tfidf.transform(text)
    combined = hstack([csr_matrix(structured.values.astype(float)), text_vec]).tocsr()

    pred_idx = model.predict(combined)
    pred_labels = label_encoder.inverse_transform(pred_idx)

    results = []
    proba_matrix = model.predict_proba(combined) if hasattr(model, "predict_proba") else None
    for i, label in enumerate(pred_labels):
        item = {"offer_du_doan": label}
        if proba_matrix is not None:
            for j, p in enumerate(proba_matrix[i]):
                item[f"xac_suat_{label_encoder.classes_[j]}"] = float(p)
        results.append(item)
    return results


@app.get("/")
def root():
    return {"status": "ok", "message": "Telco Next Best Offer API dang chay"}


@app.post("/predict")
def predict(customer: Customer):
    df = pd.DataFrame([customer.dict()])
    result = preprocess_and_predict(df)[0]
    # Endpoint 1 khach hang giu nguyen dang xac_suat long thay vi flatten
    flat_keys = [k for k in result if k.startswith("xac_suat_")]
    if flat_keys:
        result["xac_suat"] = {k.replace("xac_suat_", ""): result.pop(k) for k in flat_keys}
    return result


@app.post("/predict_batch")
async def predict_batch(
    file: UploadFile = File(...),
    output_format: str = Query("json", enum=["json", "csv"])
):
    """Nhan 1 file CSV chua nhieu khach hang (chua co cot Offers), tra ve du doan cho tung dong.
    output_format=json (mac dinh) tra ve JSON, output_format=csv tra ve file CSV tai ve truc tiep."""
    content = await file.read()
    df = pd.read_csv(io.BytesIO(content))

    predictions = preprocess_and_predict(df)
    pred_df = pd.DataFrame(predictions)

    result_df = pd.concat([df.reset_index(drop=True), pred_df], axis=1)

    if output_format == "csv":
        buffer = io.StringIO()
        result_df.to_csv(buffer, index=False, encoding="utf-8-sig")
        buffer.seek(0)
        return StreamingResponse(
            iter([buffer.getvalue()]),
            media_type="text/csv",
            headers={"Content-Disposition": "attachment; filename=ket_qua_du_doan.csv"}
        )

    return {"so_luong": len(result_df), "ket_qua": result_df.to_dict(orient="records")}
'''

with open(f'{deploy_dir}/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("Đã tạo app.py (FastAPI) với 2 endpoint: /predict (1 khách hàng) và /predict_batch (upload CSV, hỗ trợ trả JSON hoặc CSV)")


In [ ]:
# Sinh requirements.txt và README.md cho việc deploy
requirements = '''fastapi
uvicorn
python-multipart
scikit-learn
pandas
scipy
joblib
xgboost
'''

with open(f'{deploy_dir}/requirements.txt', 'w') as f:
    f.write(requirements)

readme = f'''# Telco Next Best Offer API

## Chay local
```
pip install -r requirements.txt
uvicorn app:app --reload
```
Sau do goi API:
- POST http://localhost:8000/predict (1 khach hang, gui JSON)
- POST http://localhost:8000/predict_batch (nhieu khach hang, upload file CSV)

## Deploy len Render (co san Dockerfile)
1. Push thu muc nay len 1 GitHub repo
2. Tren Render Dashboard: New -> Web Service -> chon repo vua tao
3. Render tu dong nhan Dockerfile, chon san Runtime: Docker (khong can nhap Build/Start Command)
4. Instance Type: Free -> Create Web Service
5. Doi build xong, se co URL dang https://ten-service.onrender.com
'''

with open(f'{deploy_dir}/README.md', 'w', encoding='utf-8') as f:
    f.write(readme)

print("Đã tạo requirements.txt và README.md")

### Sinh Dockerfile (dùng để deploy lên Render)

Render tự động phát hiện `Dockerfile` trong repo và build/chạy container — không cần khai báo Build Command/Start Command thủ công. Render cấp cổng qua biến môi trường `$PORT` (thường là 10000), nên Dockerfile đọc cổng từ biến này thay vì cố định.

In [ ]:
dockerfile_content = '''FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# Render cap cong qua bien moi truong PORT (mac dinh 10000 neu khong duoc set)
ENV PORT=10000
EXPOSE 10000

CMD uvicorn app:app --host 0.0.0.0 --port $PORT
'''

with open(f'{deploy_dir}/Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print("Đã tạo Dockerfile (dùng để deploy lên Render)")

In [ ]:
# Nén toàn bộ thư mục thành file .zip và tải về
shutil.make_archive('telco_nbo_deploy', 'zip', deploy_dir)

from google.colab import files
files.download('telco_nbo_deploy.zip')
print("Đã tải file 'telco_nbo_deploy.zip' — giải nén để lấy đầy đủ file phục vụ deploy.")